In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Examination"
)

xpt_file = DATA_DIR / "BMX_I.xpt"
csv_file = DATA_DIR / "BMX_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 BMX_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BMDSTATS",
    "BMXWT",
    "BMIWT",
    "BMXRECUM",
    "BMIRECUM",
    "BMXHEAD",
    "BMIHEAD",
    "BMXHT",
    "BMIHT",
    "BMXBMI",
    "BMDBMIC",
    "BMXLEG",
    "BMILEG",
    "BMXARML",
    "BMIARML",
    "BMXARMC",
    "BMIARMC",
    "BMXWAIST",
    "BMIWAIST",
    "BMXSAD1",
    "BMXSAD2",
    "BMXSAD3",
    "BMXSAD4",
    "BMDAVSAD",
    "BMDSADCM"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "BMX_I column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 9544:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 26:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,544 rows x 26 columns."
)


# ============================================================
# 4. DEFINE GENUINE MEASUREMENT DECIMAL FIELDS
# ============================================================

#
# IMPORTANT:
#
# These 14 variables are genuine continuous measurements.
#
# They MUST remain floating point.
#
# DO NOT:
#
#     round()
#
# DO NOT:
#
#     astype(Int64)
#
# DO NOT:
#
#     globally remove ".0"
#
# DO NOT:
#
#     use float_format=
#
# on these variables.
#

DECIMAL_COLUMNS = [
    "BMXWT",
    "BMXRECUM",
    "BMXHEAD",
    "BMXHT",
    "BMXBMI",
    "BMXLEG",
    "BMXARML",
    "BMXARMC",
    "BMXWAIST",
    "BMXSAD1",
    "BMXSAD2",
    "BMXSAD3",
    "BMXSAD4",
    "BMDAVSAD"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(DECIMAL_COLUMNS) != 14:

    raise ValueError(
        "Expected exactly 14 genuine decimal fields."
    )


if len(INTEGER_COLUMNS) != 12:

    raise ValueError(
        "Expected exactly 12 whole-number/code fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine measurement decimal fields:",
    len(DECIMAL_COLUMNS)
)

print(
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)

print(
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation:
#
#     5.397605346934028e-79
#
# Attached BMX_I contains NONE.
#
# Therefore no tiny -> zero repair should be performed.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 26 BMX_I variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


if tiny_total > 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in BMX_I."
    )


if true_zero_total > 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected ordinary zero values "
        "found in BMX_I."
    )


print(
    "PASS: Attached BMX_I contains "
    "0 tiny artifacts and 0 ordinary zeros."
)


# ============================================================
# 6. NO ZERO RESTORATION REQUIRED
# ============================================================

#
# Unlike DEMO_I / Dietary files, BMX_I has no
# tiny-zero artifact in this attachment.
#
# Therefore source values require NO numeric repair.
#

expected_df = source_df.copy()


print(
    "\nPASS: No tiny-value restoration required."
)


# ============================================================
# 7. DETECT GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# Run BEFORE any Int64 conversion.
#
# np.round() is used ONLY for detecting whether a value
# has a fractional component.
#
# NO dataframe value is rounded.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


detected_decimal_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{fractional_count:,} fractional observations"
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Measurement decimal pattern differs.\n\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Actual:\n{detected_decimal_columns}"
    )


print(
    "\nPASS: Exactly 14 genuine measurement "
    "decimal variables detected."
)


# ============================================================
# 8. DECIMAL FIELD DETAILS
# ============================================================

print(
    "\n--- GENUINE MEASUREMENT DETAILS ---"
)


for col in DECIMAL_COLUMNS:

    s = df[col]

    fractional_count = fractional_counts[col]

    nonmissing = int(
        s.notna().sum()
    )

    whole_looking = (
        nonmissing
        -
        fractional_count
    )


    print(
        f"\n{col}"
    )

    print(
        "  Nonmissing:",
        f"{nonmissing:,}"
    )

    print(
        "  Missing:",
        f"{int(s.isna().sum()):,}"
    )

    print(
        "  Minimum:",
        repr(float(s.min()))
    )

    print(
        "  Maximum:",
        repr(float(s.max()))
    )

    print(
        "  Fractional observations:",
        f"{fractional_count:,}"
    )

    print(
        "  Whole-looking observations:",
        f"{whole_looking:,}"
    )


# ============================================================
# 9. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 10. CONVERT ONLY PROVEN WHOLE-NUMBER FIELDS
# ============================================================

#
# Convert:
#
#     SEQN
#     status/comment/category fields
#
# to nullable Int64.
#
# DO NOT convert the 14 measurement fields.
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: 12 whole-number/code fields "
    "converted to nullable Int64."
)

print(
    "PASS: All 14 genuine body measurements "
    "remain floating-point and unchanged."
)


# ============================================================
# 11. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 12. HELPER - CONTINUOUS MEASUREMENT CHECK
# ============================================================

#
# Range checking is validation only.
#
# The measurement itself is NEVER changed.
#

def check_measurement(
    column,
    expected_min,
    expected_max,
    expected_nonmissing,
    expected_missing
):

    s = df[column]


    actual_nonmissing = int(
        s.notna()
        .sum()
    )

    actual_missing = int(
        s.isna()
        .sum()
    )

    actual_min = float(
        s.min()
    )

    actual_max = float(
        s.max()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Nonmissing:",
        f"{actual_nonmissing:,}"
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )

    print(
        "Minimum:",
        repr(actual_min)
    )

    print(
        "Maximum:",
        repr(actual_max)
    )


    if actual_nonmissing != expected_nonmissing:

        raise ValueError(
            f"{column}: nonmissing count differs."
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column}: missing count differs."
        )


    #
    # CDC endpoints for BMX_I are displayed at
    # measurement precision.
    #
    # np.isclose is used ONLY for range validation.
    # The dataframe is not rounded.
    #

    if not np.isclose(
        actual_min,
        expected_min,
        rtol=0,
        atol=1e-12
    ):

        raise ValueError(
            f"{column}: minimum differs."
        )


    if not np.isclose(
        actual_max,
        expected_max,
        rtol=0,
        atol=1e-12
    ):

        raise ValueError(
            f"{column}: maximum differs."
        )


    print(
        f"PASS: {column} range/count validated."
    )


# ============================================================
# 13. CDC CHECK - BMDSTATS
# ============================================================

#
# Body Measures Component Status:
#
# 1 Complete                 8,687
# 2 Height + weight only       366
# 3 Other partial exam         411
# 4 No body measures            80
#

check_frequency(
    "BMDSTATS",
    {
        1: 8687,
        2: 366,
        3: 411,
        4: 80
    },
    0
)


# ============================================================
# 14. CDC CHECK - BMXWT
# ============================================================

#
# Weight in kilograms.
#

check_measurement(
    "BMXWT",
    expected_min=3.6,
    expected_max=198.9,
    expected_nonmissing=9445,
    expected_missing=99
)


# ============================================================
# 15. CDC CHECK - BMIWT
# ============================================================

#
# Weight comment:
#
# 1 Could not obtain
# 3 Clothing
# 4 Medical appliance
#

check_frequency(
    "BMIWT",
    {
        1: 14,
        3: 406,
        4: 23
    },
    9101
)


# ============================================================
# 16. CDC CHECK - BMXRECUM
# ============================================================

#
# Recumbent length, cm.
#

check_measurement(
    "BMXRECUM",
    expected_min=49.3,
    expected_max=116.7,
    expected_nonmissing=1073,
    expected_missing=8471
)


# ============================================================
# 17. CDC CHECK - BMIRECUM
# ============================================================

check_frequency(
    "BMIRECUM",
    {
        1: 33
    },
    9511
)


# ============================================================
# 18. CDC CHECK - BMXHEAD
# ============================================================

check_measurement(
    "BMXHEAD",
    expected_min=36.2,
    expected_max=48.5,
    expected_nonmissing=215,
    expected_missing=9329
)


# ============================================================
# 19. CDC CHECK - BMIHEAD
# ============================================================

#
# CDC has zero populated head-circumference comments.
#

check_frequency(
    "BMIHEAD",
    {},
    9544
)


# ============================================================
# 20. CDC CHECK - BMXHT
# ============================================================

#
# Standing height, cm.
#

check_measurement(
    "BMXHT",
    expected_min=80.7,
    expected_max=202.7,
    expected_nonmissing=8769,
    expected_missing=775
)


# ============================================================
# 21. CDC CHECK - BMIHT
# ============================================================

check_frequency(
    "BMIHT",
    {
        1: 37,
        3: 68
    },
    9439
)


# ============================================================
# 22. CDC CHECK - BMXBMI
# ============================================================

#
# BMI kg/m^2.
#
# CDC created BMXBMI from measured weight and height
# and released it to one decimal place.
#
# IMPORTANT:
#
# We DO NOT recalculate or round BMXBMI here.
# We preserve the released value.
#

check_measurement(
    "BMXBMI",
    expected_min=11.5,
    expected_max=67.3,
    expected_nonmissing=8756,
    expected_missing=788
)


# ============================================================
# 23. CDC CHECK - BMDBMIC
# ============================================================

#
# Child/youth BMI category:
#
# 1 Underweight
# 2 Normal
# 3 Overweight
# 4 Obese
#

check_frequency(
    "BMDBMIC",
    {
        1: 86,
        2: 2041,
        3: 561,
        4: 652
    },
    6204
)


# ============================================================
# 24. CDC CHECK - BMXLEG
# ============================================================

check_measurement(
    "BMXLEG",
    expected_min=24.0,
    expected_max=51.5,
    expected_nonmissing=7110,
    expected_missing=2434
)


# ============================================================
# 25. CDC CHECK - BMILEG
# ============================================================

check_frequency(
    "BMILEG",
    {
        1: 402
    },
    9142
)


# ============================================================
# 26. CDC CHECK - BMXARML
# ============================================================

check_measurement(
    "BMXARML",
    expected_min=10.0,
    expected_max=47.4,
    expected_nonmissing=8976,
    expected_missing=568
)


# ============================================================
# 27. CDC CHECK - BMIARML
# ============================================================

check_frequency(
    "BMIARML",
    {
        1: 420
    },
    9124
)


# ============================================================
# 28. CDC CHECK - BMXARMC
# ============================================================

check_measurement(
    "BMXARMC",
    expected_min=10.1,
    expected_max=58.4,
    expected_nonmissing=8976,
    expected_missing=568
)


# ============================================================
# 29. CDC CHECK - BMIARMC
# ============================================================

check_frequency(
    "BMIARMC",
    {
        1: 421
    },
    9123
)


# ============================================================
# 30. CDC CHECK - BMXWAIST
# ============================================================

check_measurement(
    "BMXWAIST",
    expected_min=40.0,
    expected_max=171.6,
    expected_nonmissing=8313,
    expected_missing=1231
)


# ============================================================
# 31. CDC CHECK - BMIWAIST
# ============================================================

check_frequency(
    "BMIWAIST",
    {
        1: 489
    },
    9055
)


# ============================================================
# 32. CDC CHECK - SAGITTAL ABDOMINAL DIAMETERS
# ============================================================

check_measurement(
    "BMXSAD1",
    expected_min=9.5,
    expected_max=40.7,
    expected_nonmissing=6983,
    expected_missing=2561
)


check_measurement(
    "BMXSAD2",
    expected_min=9.7,
    expected_max=40.9,
    expected_nonmissing=6983,
    expected_missing=2561
)


check_measurement(
    "BMXSAD3",
    expected_min=11.3,
    expected_max=39.5,
    expected_nonmissing=353,
    expected_missing=9191
)


check_measurement(
    "BMXSAD4",
    expected_min=11.1,
    expected_max=39.8,
    expected_nonmissing=353,
    expected_missing=9191
)


check_measurement(
    "BMDAVSAD",
    expected_min=9.6,
    expected_max=40.8,
    expected_nonmissing=6983,
    expected_missing=2561
)


# ============================================================
# 33. CDC CHECK - BMDSADCM
# ============================================================

check_frequency(
    "BMDSADCM",
    {
        1: 436,
        2: 2,
        3: 3,
        4: 1,
        5: 4
    },
    9098
)


# ============================================================
# 34. SAD STRUCTURE CHECK
# ============================================================

#
# CDC notes:
#
# Most participants have readings:
#
#     BMXSAD1
#     BMXSAD2
#
# Some have readings 3 and 4 when additional
# measurements were required.
#
# BMDAVSAD is the released calculated average.
#
# We do NOT recalculate or overwrite BMDAVSAD.
#

sad1_present = (
    df["BMXSAD1"]
    .notna()
    .to_numpy(dtype=bool)
)

sad2_present = (
    df["BMXSAD2"]
    .notna()
    .to_numpy(dtype=bool)
)

average_present = (
    df["BMDAVSAD"]
    .notna()
    .to_numpy(dtype=bool)
)


sad12_mismatch = int(
    np.sum(
        sad1_present
        !=
        sad2_present
    )
)


sad_average_mismatch = int(
    np.sum(
        sad1_present
        !=
        average_present
    )
)


print(
    "\n--- SAD STRUCTURE ---"
)

print(
    "SAD1/SAD2 mask mismatches:",
    sad12_mismatch
)

print(
    "SAD1/BMDAVSAD mask mismatches:",
    sad_average_mismatch
)


if sad12_mismatch != 0:

    raise ValueError(
        "BMXSAD1 and BMXSAD2 missing masks differ."
    )


if sad_average_mismatch != 0:

    raise ValueError(
        "BMDAVSAD availability does not match SAD1."
    )


print(
    "PASS: SAD measurement structure validated."
)


# ============================================================
# 35. CONFIRM DECIMAL FIELDS REMAIN FLOATING POINT
# ============================================================

decimal_dtype_failures = []


for col in DECIMAL_COLUMNS:

    if not pd.api.types.is_float_dtype(
        df[col].dtype
    ):

        decimal_dtype_failures.append(
            col
        )


if decimal_dtype_failures:

    raise ValueError(
        "Some genuine measurement fields "
        "are no longer floating-point:\n"
        + str(decimal_dtype_failures)
    )


print(
    "\nPASS: All 14 genuine measurement fields "
    "remain floating-point."
)


# ============================================================
# 36. CONFIRM NO ZEROS CREATED
# ============================================================

#
# Original BMX_I contained:
#
#     tiny values     = 0
#     ordinary zeros = 0
#
# Therefore conversion should not create any zeros.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK ---"
)

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zeros were created:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No zero values were created."
)


# ============================================================
# 37. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# BMX_I required NO numeric repair.
#
# Therefore every numeric value must remain exactly
# numerically equal to the source XPT.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: All 26 variables preserve "
    "the source XPT values exactly."
)


# ============================================================
# 38. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# This keeps:
#
#     BMXWT
#     BMXHT
#     BMXBMI
#     BMXWAIST
#     SAD measurements
#
# exactly as released.
#
# This is also the same rule we use for your genuine
# Dietary decimals.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 39. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 40. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: 0 numeric differences across "
    "all 26 variables."
)


# ============================================================
# 41. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 42. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 43. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT values found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 44. .0 CHECK - INTEGER/CODE FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# BMX_I is a MIXED decimal/integer file.
#
# DO NOT search all columns for ".0".
#
# A measurement such as:
#
#     BMXWT    = 75.0
#     BMXHT    = 170.0
#     BMXBMI   = 25.0
#     BMXWAIST = 90.0
#
# is legitimate.
#
# Only inspect the 12 integer/code fields.
#

print(
    "\n--- INTEGER FIELD .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' in integer/code fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "the 12 integer/code fields."
)


# ============================================================
# 45. LEGITIMATE MEASUREMENT .0 VALUES
# ============================================================

#
# INFORMATIONAL ONLY.
#
# Whole-looking values in a genuine continuous measurement
# remain floats.
#
# Examples:
#
#     24.0
#     40.0
#     75.0
#
# These are NOT errors.
#

print(
    "\n--- LEGITIMATE MEASUREMENT .0 TOKENS ---"
)


decimal_dot_zero_counts = {}


for col in DECIMAL_COLUMNS:

    count = int(
        (
            raw_csv_df[col].ne("")
            &
            raw_csv_df[col].str.endswith(".0")
        )
        .sum()
    )


    decimal_dot_zero_counts[col] = count


    print(
        f"{col}:",
        f"{count:,}"
    )


print(
    "PASS: Measurement '.0' values "
    "are intentionally preserved."
)


# ============================================================
# 46. EXACT MEASUREMENT DECIMAL FIDELITY
# ============================================================

#
# Most important continuous-value check.
#
# rtol = 0
# atol = 0
#
# No numeric difference is accepted.
#

print(
    "\n--- EXACT MEASUREMENT DECIMAL FIDELITY ---"
)


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Measurement decimal fidelity failed:\n"
        + str(decimal_failures)
    )


print(
    "PASS: All 14 genuine measurement variables "
    "preserved exactly."
)


# ============================================================
# 47. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values found in CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 48. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 49. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 50. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "BMX_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)

print(
    "Component: Examination - Body Measures"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    tiny_total
)

print(
    "Original ordinary zeros:",
    true_zero_total
)

print(
    "Validated zero restorations:",
    "NONE"
)

print(
    "Genuine measurement decimal variables:",
    len(DECIMAL_COLUMNS)
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)

print(
    "BMXWT nonmissing:",
    int(
        df["BMXWT"].notna().sum()
    )
)

print(
    "BMXHT nonmissing:",
    int(
        df["BMXHT"].notna().sum()
    )
)

print(
    "BMXBMI nonmissing:",
    int(
        df["BMXBMI"].notna().sum()
    )
)

print(
    "BMXWAIST nonmissing:",
    int(
        df["BMXWAIST"].notna().sum()
    )
)

print(
    "BMDAVSAD nonmissing:",
    int(
        df["BMDAVSAD"].notna().sum()
    )
)

print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)

print(
    "Missing-position differences:",
    "0 expected"
)

print(
    "Measurement decimal differences:",
    "0 expected"
)

print(
    "Tiny values remaining:",
    "0 expected"
)

print(
    "Unnecessary '.0' in integer fields:",
    "0 expected"
)

print(
    "Measurement '.0' values:",
    "ALLOWED / PRESERVED"
)

print(
    "Scientific notation:",
    "0 expected"
)

print(
    "\nFINAL RULE:"
)

print(
    "All genuine BMX measurement decimals remain "
    "exactly AS IS."
)

print(
    "No measurement value was rounded or rewritten."
)

NHANES 2015-2016 BMX_I XPT -> CSV
Rows: 9,544
Columns: 26
PASS: Structure = 9,544 rows x 26 columns.

--- FIELD TYPE PLAN ---
Genuine measurement decimal fields: 14
['BMXWT', 'BMXRECUM', 'BMXHEAD', 'BMXHT', 'BMXBMI', 'BMXLEG', 'BMXARML', 'BMXARMC', 'BMXWAIST', 'BMXSAD1', 'BMXSAD2', 'BMXSAD3', 'BMXSAD4', 'BMDAVSAD']
Whole-number/code fields: 12
['SEQN', 'BMDSTATS', 'BMIWT', 'BMIRECUM', 'BMIHEAD', 'BMIHT', 'BMDBMIC', 'BMILEG', 'BMIARML', 'BMIARMC', 'BMIWAIST', 'BMDSADCM']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: Attached BMX_I contains 0 tiny artifacts and 0 ordinary zeros.

PASS: No tiny-value restoration required.

--- GENUINE DECIMAL DETECTION ---
BMXWT: 8,482 fractional observations
BMXRECUM: 972 fractional observations
BMXHEAD: 196 fractional observations
BMXHT: 7,909 fractional observations
BMXBMI: 7,868 fractional observations
BMXLEG: 5,637 fractional observations
BMXARML: 6,455 fractional observations
BMXARMC: 7,652 fractional obser